<a href="https://colab.research.google.com/github/dipanwitad-create/NASSCOM_FDP_programme/blob/main/day2Calculus_for_ML_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
U5 — Calculus for ML: Lab
How models learn — by following the gradient downhill — derivatives · partial derivatives · the gradient · chain rule · backpropagation · Hessian

Day 2 · Phase B — Mathematical Foundations · Bridge unit (Module 1.3.8)

objectives
By the end of this lab you will be able to:

Compute derivatives numerically (finite difference) and symbolically (SymPy)

Take partial derivatives and assemble the gradient of a multivariable function

Apply the chain rule by hand and verify it with SymPy

Run a forward and backward pass through a 2-layer network and derive its gradients

Compute a Hessian, and see how a gradient-descent step uses the gradient

how to use this lab
Each section has two kinds of cells:

Worked demo cells — run them top to bottom and read the comments to learn the pattern.

LAB EXERCISE cells (marked 🧪) — your turn. Replace each # YOUR CODE HERE with working code.

Run cells with Shift + Enter. Run the demos before attempting the exercises.

In [1]:
# Core imports for the whole lab
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')      # symbolic variables we'll reuse
sp.init_printing()            # pretty-print symbolic math
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)

Setup complete. SymPy 1.14.0 | NumPy 2.0.2


In [2]:
# -----------------------------------------------------------
# 🔹 1A. NUMERICAL DERIVATIVE (finite difference)
# -----------------------------------------------------------

# The derivative is the slope: how much f changes for a tiny step h
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')


Numerical f'(3): 6.0  (exact = 6)


 **LAB EXERCISE 1 **— Numerical vs symbolic derivative
For the function g(x) = x**3 + 2*x:

Compute the numerical derivative at x = 2 using a finite difference.
Compute the symbolic derivative with sp.diff and print it.
Evaluate the symbolic derivative at x = 2 and confirm it matches step 1.

In [4]:
import numpy as np
import sympy as sp

def g(x):
    return x**3 + 2*x

# 1. Numerical derivative at x = 2
h = 1e-6
x0 = 2

numerical_derivative = (g(x0 + h) - g(x0)) / h

print("Numerical Derivative at x=2:")
print(numerical_derivative)

# 2. Symbolic derivative
x = sp.symbols('x')
expr = x**3 + 2*x

symbolic_derivative = sp.diff(expr, x)

print("\nSymbolic Derivative:")
print(symbolic_derivative)

# 3. Evaluate symbolic derivative at x = 2
symbolic_value = symbolic_derivative.subs(x, 2)

print("\nSymbolic Derivative at x=2:")
print(symbolic_value)

# Compare
print("\nDo they match?")
print(np.isclose(numerical_derivative, float(symbolic_value)))


Numerical Derivative at x=2:
14.000006002490295

Symbolic Derivative:
3*x**2 + 2

Symbolic Derivative at x=2:
14

Do they match?
True


In [5]:
# -----------------------------------------------------------
# 🔹 2A. PARTIAL DERIVATIVES
# -----------------------------------------------------------

f2 = x**2 + 3*x*y + y**2

# A partial derivative differentiates ONE variable, holding others fixed
print('df/dx =', sp.diff(f2, x))     # 2*x + 3*y
print('df/dy =', sp.diff(f2, y))     # 3*x + 2*y

df/dx = 2*x + 3*y
df/dy = 3*x + 2*y


In [6]:
# -----------------------------------------------------------
# 🔹 2B. THE GRADIENT (vector of partials)
# -----------------------------------------------------------

# The gradient stacks every partial derivative into one vector
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)

# Evaluate the gradient at the point (x=1, y=2)
grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)   # points in the steepest-ascent direction

grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


In [7]:
import sympy as sp

# Define symbols
x, y = sp.symbols('x y')

# Function
h2 = x**2 * y + sp.sin(y)

# 1. dh/dx and dh/dy
dh_dx = sp.diff(h2, x)
dh_dy = sp.diff(h2, y)

print("dh/dx =", dh_dx)
print("dh/dy =", dh_dy)

# 2. Assemble the gradient list
gradient = [dh_dx, dh_dy]

print("\nGradient:")
print(gradient)

# 3. Evaluate at (x=2, y=0)
gradient_at_point = [g.subs({x: 2, y: 0}) for g in gradient]

print("\nGradient at (2,0):")
print(gradient_at_point)

dh/dx = 2*x*y
dh/dy = x**2 + cos(y)

Gradient:
[2*x*y, x**2 + cos(y)]

Gradient at (2,0):
[0, 5]


In [8]:
# -----------------------------------------------------------
# 🔹 3A. CHAIN RULE BY HAND vs SymPy
# -----------------------------------------------------------

# y = sin(x**2) is a composition: outer = sin(u), inner = u = x**2
# Chain rule:  dy/dx = cos(u) * du/dx = cos(x**2) * 2x
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)

By hand : 2*x*cos(x**2)
By SymPy: 2*x*cos(x**2)
Match?   True


In [9]:
# -----------------------------------------------------------
# 🔹 3B. CHAINING THREE FUNCTIONS
# -----------------------------------------------------------

# y = (3x + 1)**4  -> outer^4, inner (3x+1)
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))   # 12*(3x+1)^3

d/dx (3x+1)^4 = 12*(3*x + 1)**3


In [10]:
import sympy as sp

# Define symbol
x = sp.symbols('x')

# Expression
y = sp.exp(x**2 + 1)

# 1. By hand using Chain Rule
inner = x**2 + 1
inner_derivative = 2*x

by_hand = sp.exp(inner) * inner_derivative

print("By Hand:")
print(by_hand)

# 2. Using sp.diff
by_sympy = sp.diff(y, x)

print("\nUsing sp.diff:")
print(by_sympy)

# 3. Confirm they match
print("\nDo they match?")
print(sp.simplify(by_hand - by_sympy) == 0)

By Hand:
2*x*exp(x**2 + 1)

Using sp.diff:
2*x*exp(x**2 + 1)

Do they match?
True


In [11]:
# -----------------------------------------------------------
# 🔹 4A. FORWARD PASS
# -----------------------------------------------------------

# Tiny toy problem: 4 samples, 3 input features, 5 hidden units, 1 output
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1                 # linear layer 1
h  = np.maximum(0, z1)      # ReLU activation
y_hat = h @ W2              # linear layer 2 (prediction)
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))

Initial loss: 1.6936


In [12]:
# -----------------------------------------------------------
# 🔹 4B. BACKWARD PASS (gradients via the chain rule)
# -----------------------------------------------------------

# Work backwards from the loss, one link at a time
dy   = 2 * (y_hat - Y) / Y.size      # d loss / d y_hat
dW2  = h.T @ dy                      # d loss / d W2
dh   = dy @ W2.T                     # d loss / d h
dz1  = dh * (z1 > 0)                 # ReLU gradient (1 where z1>0 else 0)
dW1  = X.T @ dz1                     # d loss / d W1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')

dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)


In [13]:
# -----------------------------------------------------------
# 🔹 4C. ONE GRADIENT-DESCENT STEP SHOULD LOWER THE LOSS
# -----------------------------------------------------------

lr = 0.1
W1 -= lr * dW1               # step downhill
W2 -= lr * dW2

# Recompute the loss after the update
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

Loss before: 1.6936
Loss after : 1.6524 -> should be lower


In [14]:
import numpy as np

# Data and weights
np.random.seed(42)

Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)

Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output

# --------------------------------------------------
# 1. Forward Pass
# --------------------------------------------------

# Hidden layer pre-activation
z1 = Xb @ Wa

# ReLU activation
h = np.maximum(0, z1)

# Output layer
y_hat = h @ Wb

# Mean Squared Error Loss
loss_before = np.mean((y_hat - Yb) ** 2)

print("Loss before update:", loss_before)

# --------------------------------------------------
# 2. Backward Pass
# --------------------------------------------------

n = Xb.shape[0]

# dLoss/dy_hat
dy = (2/n) * (y_hat - Yb)

# dLoss/dWb
dWb = h.T @ dy

# dLoss/dh
dh = dy @ Wb.T

# ReLU derivative
dz1 = dh * (z1 > 0)

# dLoss/dWa
dWa = Xb.T @ dz1

# --------------------------------------------------
# 3. Gradient Descent Step
# --------------------------------------------------

lr = 0.05

Wa_new = Wa - lr * dWa
Wb_new = Wb - lr * dWb

# Forward pass again using updated weights
z1_new = Xb @ Wa_new
h_new = np.maximum(0, z1_new)
y_hat_new = h_new @ Wb_new

loss_after = np.mean((y_hat_new - Yb) ** 2)

print("Loss after update :", loss_after)

Loss before update: 0.3798212978770168
Loss after update : 0.3775535471844222


In [15]:
# -----------------------------------------------------------
# 🔹 5A. THE HESSIAN (matrix of second derivatives = curvature)
# -----------------------------------------------------------

f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)        # [[2, 3], [3, 2]]

ValueError: 
Can't calculate derivative wrt exp(x**2 + 1).

In [16]:
import sympy as sp

# Define symbols first
x, y = sp.symbols('x y')

# Function
f5 = x**2 + 3*x*y + y**2

# Hessian matrix
H = sp.hessian(f5, (x, y))

print('Hessian of f:')
sp.pprint(H)

Hessian of f:
⎡2  3⎤
⎢    ⎥
⎣3  2⎦


In [17]:
import sympy as sp

# Define symbols first
x, y = sp.symbols('x y')

# Function
f5 = x**2 + 3*x*y + y**2

# Hessian matrix
H = sp.hessian(f5, (x, y))

print('Hessian of f:')
sp.pprint(H)

Hessian of f:
⎡2  3⎤
⎢    ⎥
⎣3  2⎦


LAB EXERCISE 5 — Hessian + gradient descent
Compute the Hessian of f = x**4 + y**2 with sp.hessian.
Run gradient descent to minimise f(x) = (x - 7)**2: start at x = 0, lr = 0.1, 20 steps, using f'(x) = 2*(x - 7). Print the final x (should approach 7).

In [18]:
import sympy as sp

# Define symbols
x, y = sp.symbols('x y')

# --------------------------------------------------
# 1. Hessian of f = x^4 + y^2
# --------------------------------------------------

f = x**4 + y**2

H = sp.hessian(f, (x, y))

print("Hessian of f:")
sp.pprint(H)

# --------------------------------------------------
# 2. Gradient Descent to minimize (x - 7)^2
# --------------------------------------------------

xv = 0.0
lr = 0.1

for step in range(20):
    grad = 2 * (xv - 7)      # derivative of (x - 7)^2
    xv = xv - lr * grad

print("\nFinal x:", xv)

Hessian of f:
⎡    2   ⎤
⎢12⋅x   0⎥
⎢        ⎥
⎣  0    2⎦

Final x: 6.91929549467752
